## Arguments

In [1]:
VIDEOS_DIR = "C:/Users/chany/Downloads/helipad_250221"
FRAME_INTERVAL = 30
MULTITHREAD = True
SERVER_MACHINE_1 = "chan@223.130.138.39"
SERVER_MACHINE_2 = "chan@223.130.152.210"
SERVER_DATA_PATH = "/home/chan/data"
SAMPLING_DENSITY = 1

## Setup

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys

dir_base = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(dir_base)

from utils.save_every_nth_frames_from_video import save_frames_from_videos
from utils.modify_registration import modify_registration

OUTPUT_DIR = os.path.realpath(os.path.join(VIDEOS_DIR, "data_"+os.path.basename(VIDEOS_DIR)))
VIDEOS_DIR = os.path.realpath(VIDEOS_DIR)
IMAGES_DIR = os.path.join(VIDEOS_DIR, "images")

def run(command):
    print(command)
    os.system(command)

## 1. Videos -> Images

In [3]:
save_frames_from_videos(
    videos_folder_path=VIDEOS_DIR,
    frame_interval=FRAME_INTERVAL,
    multithread=MULTITHREAD
)

Press 'q' to stop the process.


## 2. SFM

In [4]:
run(f"""{
    os.path.join(
        dir_base,
        "reality_capture",
        "RC_SFM.BAT"
    )
} {VIDEOS_DIR}""")

c:\Users\chany\Documents\SWAROBO\reconstruction\reality_capture\RC_SFM.BAT C:\Users\chany\Downloads\helipad_250221


## 3. Modify registration

In [ ]:
modify_registration(OUTPUT_DIR)

File paths have been updated successfully.


## 4. Transfer to server

In [10]:
for server_machine, folder_dir in [(SERVER_MACHINE_1, OUTPUT_DIR), (SERVER_MACHINE_2, IMAGES_DIR)]:
    SERVER_DATA_ADDED_PATH = SERVER_DATA_PATH+"/"+os.path.basename(OUTPUT_DIR)
    run(f"ssh {server_machine} mkdir -p {SERVER_DATA_ADDED_PATH}")
    run(f"scp -vr {folder_dir}/* {server_machine}:{SERVER_DATA_ADDED_PATH}/")
    run(f"scp -vr {os.path.join(dir_base, 'scripts', 'server-scripts')} {server_machine}:/home/chan/")

ssh chan@223.130.138.39 mkdir -p /home/chan/data/incheon_250206_car_3
scp -vr C:/Users/chany/Downloads/incheon_250206_car_3/* chan@223.130.138.39:/home/chan/data/incheon_250206_car_3/
scp -vr c:\Users\chany\Documents\SWAROBO\reconstruction\scripts\server-scripts chan@223.130.138.39:/home/chan/
ssh chan@223.130.152.210 mkdir -p /home/chan/data/incheon_250206_car_3
scp -vr C:/Users/chany/Downloads/incheon_250206_car_3/images/* chan@223.130.152.210:/home/chan/data/incheon_250206_car_3/
scp -vr c:\Users\chany\Documents\SWAROBO\reconstruction\scripts\server-scripts chan@223.130.152.210:/home/chan/


## 5. Start training process

In [6]:
for server_machine, colmap in [(SERVER_MACHINE_1, "False"), (SERVER_MACHINE_2, "True")]:
    command = f"ssh {server_machine} bash /home/chan/server-scripts/run.bash \
        {SERVER_DATA_ADDED_PATH} \
        {colmap} \
        {SAMPLING_DENSITY}"
    run(command)

ssh chan@223.130.138.39 bash /home/chan/server-scripts/run.bash         /home/chan/data/data_goldhome250218         False         1
ssh chan@223.130.152.210 bash /home/chan/server-scripts/run.bash         /home/chan/data/data_goldhome250218         True         1
